# Backtest: do fans or experts see what draft slot doesn't?

The design from the README, executed:

| Classes | Role |
| --- | --- |
| 2021–2024 | primary backtest |
| 2025 | robustness check (one season of outcomes) |
| 2026 | out-of-sample prediction |

The outcome is **DrAV percentile within class** (`dr_av_pct`), the rank-based
version chosen for its robustness to DrAV's heavy right skew. Both signals —
fan sentiment (`sent_z`, standardized within fanbase-class) and the expert
grade (`grade_z`, standardized within class) — enter the same way, so their
coefficients are directly comparable.

**The control.** Draft slot alone predicts outcomes, and everyone knows the
slot when they react. All models therefore include `log(pick)`; the question
is never "does sentiment correlate with success" but "does sentiment explain
success *beyond the slot*."

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PROCESSED = "../data/processed"
key = ["season", "team", "round", "pick"]

outcomes = pd.read_csv(f"{PROCESSED}/draft_outcomes_2021_2025.csv")
sent = pd.read_csv(f"{PROCESSED}/pick_sentiment.csv")
grades = pd.read_csv(f"{PROCESSED}/expert_grades.csv")

df = (outcomes
      .merge(sent[key + ["n_comments", "sent_mean", "sent_z"]], on=key, how="left")
      .merge(grades[key + ["wf_grade", "grade_points", "grade_z"]], on=key, how="left"))
df["log_pick"] = np.log(df["pick"])

print(f"outcome rows: {len(df)}")
print(f"with sentiment: {df.sent_z.notna().sum()} "
      f"(missing: too few comments or a lone pick in its team-year cell)")
print(f"with expert grade: {df.grade_z.notna().sum()}")

MIN_COMMENTS = 10
base = df[(df.sent_z.notna()) & (df.grade_z.notna())
          & (df.n_comments >= MIN_COMMENTS)].copy()
train = base[base.season <= 2024]
holdout = base[base.season == 2025]
print(f"\nanalysis sample (≥{MIN_COMMENTS} comments): "
      f"{len(train)} train (2021–24), {len(holdout)} holdout (2025)")

outcome rows: 1294
with sentiment: 1278 (missing: too few comments or a lone pick in its team-year cell)
with expert grade: 1294

analysis sample (≥10 comments): 996 train (2021–24), 251 holdout (2025)


### The head-to-head

Four nested models on 2021–2024, HC3 robust standard errors. If fans carry
signal that experts don't, `sent_z` should survive the inclusion of
`grade_z`; if fan reaction is just noisy expert consensus, it shouldn't.

In [2]:
models = {
    "slot only":        "dr_av_pct ~ log_pick",
    "slot + fans":      "dr_av_pct ~ log_pick + sent_z",
    "slot + expert":    "dr_av_pct ~ log_pick + grade_z",
    "slot + both":      "dr_av_pct ~ log_pick + sent_z + grade_z",
}
fits = {name: smf.ols(f, data=train).fit(cov_type="HC3")
        for name, f in models.items()}

rows = []
for name, fit in fits.items():
    row = {"model": name, "adj_R2": fit.rsquared_adj, "AIC": fit.aic}
    for term in ["sent_z", "grade_z"]:
        if term in fit.params:
            row[f"{term} coef"] = fit.params[term]
            row[f"{term} p"] = fit.pvalues[term]
    rows.append(row)
print(pd.DataFrame(rows).round(4).to_string(index=False))

print("\nfull summary — slot + both:")
print(fits["slot + both"].summary().tables[1])

        model  adj_R2      AIC  sent_z coef  sent_z p  grade_z coef  grade_z p
    slot only  0.3481 -88.7957          NaN       NaN           NaN        NaN
  slot + fans  0.3541 -97.1329       0.0261    0.0010           NaN        NaN
slot + expert  0.3486 -88.6944          NaN       NaN        0.0100     0.1721
  slot + both  0.3542 -96.1795       0.0251    0.0017        0.0075     0.3162

full summary — slot + both:
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.3182      0.036     37.128      0.000       1.249       1.388
log_pick      -0.1783      0.008    -23.095      0.000      -0.193      -0.163
sent_z         0.0251      0.008      3.144      0.002       0.009       0.041
grade_z        0.0075      0.007      1.002      0.316      -0.007       0.022


### Robustness

The headline number should not depend on arbitrary choices. Each variation
re-runs the joint model:

- **≥30 comments** — restrict to picks whose sentiment mean rests on a real
  sample;
- **weighted** — weight by √(comment count), acknowledging that sentiment is
  measured with error that shrinks as threads grow;
- **z-scored DrAV** — swap the rank outcome for the parametric one;
- **round-1-to-3 only** — day-1/2 picks, where both fans and Walt are paying
  closest attention;
- **2025 holdout** — one season of outcomes, the noisiest possible test:
  coefficients from 2021–24 applied to 2025, scored by correlation with the
  slot residual.

In [3]:
def joint(data, formula="dr_av_pct ~ log_pick + sent_z + grade_z", **kw):
    fit = smf.ols(formula, data=data).fit(cov_type="HC3", **kw) if not kw \
        else smf.wls(formula, data=data, weights=kw["w"]).fit(cov_type="HC3")
    return fit

variants = {
    "baseline (≥10 comments)": smf.ols(models["slot + both"], data=train)
        .fit(cov_type="HC3"),
    "≥30 comments": smf.ols(models["slot + both"],
        data=train[train.n_comments >= 30]).fit(cov_type="HC3"),
    "weighted √n": smf.wls(models["slot + both"], data=train,
        weights=np.sqrt(train.n_comments)).fit(cov_type="HC3"),
    "outcome = dr_av_z": smf.ols("dr_av_z ~ log_pick + sent_z + grade_z",
        data=train).fit(cov_type="HC3"),
    "rounds 1–3": smf.ols(models["slot + both"],
        data=train[train["round"] <= 3]).fit(cov_type="HC3"),
}
rows = []
for name, fit in variants.items():
    rows.append({
        "variant": name, "n": int(fit.nobs),
        "sent coef": fit.params.get("sent_z"), "sent p": fit.pvalues.get("sent_z"),
        "grade coef": fit.params.get("grade_z"), "grade p": fit.pvalues.get("grade_z"),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))

                variant   n  sent coef  sent p  grade coef  grade p
baseline (≥10 comments) 996     0.0251  0.0017      0.0075   0.3162
           ≥30 comments 901     0.0280  0.0013      0.0077   0.3076
            weighted √n 996     0.0222  0.0125      0.0107   0.1446
      outcome = dr_av_z 996     0.0478  0.0371      0.0620   0.0166
             rounds 1–3 409     0.0183  0.1892      0.0121   0.2380


In [4]:
# 2025 holdout: apply 2021-24 coefficients, correlate with realized residuals
slot_fit = smf.ols("dr_av_pct ~ log_pick", data=train).fit()
holdout = holdout.assign(
    resid=holdout.dr_av_pct - slot_fit.predict(holdout))

for name in ["slot + fans", "slot + expert", "slot + both"]:
    pred = fits[name].predict(holdout) - slot_fit.predict(holdout)
    r = np.corrcoef(pred, holdout.resid)[0, 1]
    print(f"{name:14} holdout residual correlation: {r:+.3f}")
print(f"\n(n = {len(holdout)} picks; one season of DrAV — treat as directional)")

slot + fans    holdout residual correlation: +0.032
slot + expert  holdout residual correlation: +0.078
slot + both    holdout residual correlation: +0.053

(n = 251 picks; one season of DrAV — treat as directional)


### The 2026 class, scored before the league can

The joint model fitted on 2021–2024 is applied to the 2026 picks. The output
is each pick's **predicted class-percentile above or below what its slot
implies** — positive means the model expects the player to outperform his
draft position. Rows resolve as DrAV accumulates over the coming seasons.

The most interesting rows are the disagreements: picks where the fanbase and
the expert pulled in opposite directions.

In [5]:
sent26 = sent[sent.season == 2026]
grades26 = grades[grades.season == 2026]
p26 = (pd.read_csv(f"{PROCESSED}/draft_2026_picks.csv")
       .merge(sent26[key + ["n_comments", "sent_z"]], on=key, how="left")
       .merge(grades26[key + ["wf_grade", "grade_z"]], on=key, how="left"))
p26["log_pick"] = np.log(p26["pick"])
pred26 = p26[(p26.sent_z.notna()) & (p26.grade_z.notna())
             & (p26.n_comments >= MIN_COMMENTS)].copy()

fit = fits["slot + both"]
pred26["pred_pct"] = fit.predict(pred26)
pred26["slot_pct"] = slot_fit.predict(pred26)
pred26["edge"] = pred26.pred_pct - pred26.slot_pct

cols = ["team", "round", "pick", "pfr_player_name", "wf_grade",
        "sent_z", "grade_z", "edge"]
print(f"2026 picks scored: {len(pred26)}")
print("\nmodel's favorite value picks:")
print(pred26.nlargest(10, "edge")[cols].round(2).to_string(index=False))
print("\nmodel's biggest concerns:")
print(pred26.nsmallest(10, "edge")[cols].round(2).to_string(index=False))

dis = pred26[np.sign(pred26.sent_z) != np.sign(pred26.grade_z)]
dis = dis.assign(gap=(dis.sent_z - dis.grade_z).abs()).nlargest(10, "gap")
print("\nfans and the expert disagree most:")
print(dis[cols].round(2).to_string(index=False))

2026 picks scored: 249

model's favorite value picks:
team  round  pick      pfr_player_name wf_grade  sent_z  grade_z  edge
 BAL      1    14     Olaivavega Ioane        A    2.24     1.00  0.07
 CAR      4   129         Will Lee III       A-    2.17     0.73  0.06
 LVR      7   229    Brandon Cleveland        B    2.38     0.10  0.06
 KAN      5   161       Emmett Johnson        A    1.97     1.00  0.06
 WAS      3    71     Antonio Williams       A-    1.75     0.73  0.05
 NWE      6   196 Dametrious Crownover        B    2.05     0.10  0.05
 CLE      7   248          Carsen Ryan        C    2.23    -0.80  0.05
 MIN      5   163     Charles Demmings       B+    1.76     0.37  0.05
 LAC      6   202         Logan Taylor        A    1.59     1.00  0.04
 DET      1    17         Blake Miller       B+    1.40     0.37  0.04

model's biggest concerns:
team  round  pick  pfr_player_name wf_grade  sent_z  grade_z  edge
 CIN      4   140     Colbie Young        D   -2.12    -1.70 -0.07
 JAX

In [6]:
out = pred26[["season"] + cols + ["n_comments", "pred_pct", "slot_pct"]]
out_path = f"{PROCESSED}/predictions_2026.csv"
out.to_csv(out_path, index=False)
print(f"wrote {len(out)} rows to {out_path}")

wrote 249 rows to ../data/processed/predictions_2026.csv
